# 07 — Dynamic Agents: Blueprint-Based Agent Creation

## What are Blueprint Agents?

Normally, a graph node references a **named agent** that is registered in the orchestrator's agent registry (e.g., `EchoAgent`, `ResumeParserAgent`). These are **static agents** — their behaviour is fully defined at deploy time.

**Blueprint agents** (also called *dynamic agents*) take a different approach: instead of referencing a pre-registered agent by name, you embed a **blueprint dictionary** directly in the node definition. The orchestrator uses this blueprint to **create a new agent instance on-the-fly** at execution time.

## Why Use Blueprints?

| Use Case | Benefit |
|----------|---------|
| Rapidly prototype new agent behaviours | No deploy cycle needed |
| LLM-generated agent specs | Graphs can self-modify |
| Tenant-specific customizations | Each tenant has unique prompts |
| A/B testing agent prompts | Swap blueprints without redeployment |

## Blueprint Structure

```python
blueprint = {
    "system_prompt": "You are a...",   # LLM system prompt
    "model": "gpt-4o",                  # optional: override default LLM
    "temperature": 0.7,                 # optional: sampling temperature
    "max_tokens": 500,                  # optional: token limit
    "tools": [...],                     # optional: tool specs
}
```

## The `create_agent_activity` Hook

When the orchestrator encounters a blueprint node, it fires the `create_agent_activity` Temporal activity. This activity:
1. Validates the blueprint schema
2. Instantiates a new agent object (wraps an LLM call with the given system prompt)
3. Registers the ephemeral agent for the duration of the workflow
4. Executes the node using this freshly created agent

In [ ]:
import time
import json
import pprint

from multigen import SyncMultigenClient
from multigen.dsl import GraphBuilder

ORCHESTRATOR_URL = "http://localhost:8009"
client = SyncMultigenClient(base_url=ORCHESTRATOR_URL, timeout=60.0)
assert client.ping(), "Orchestrator not reachable."
print("Connected.")

## 1. Define Blueprint Dictionaries

We define two blueprints: one for a specialized technical screener, another for a cultural fit assessor. Neither of these is a pre-registered agent — they are defined purely by their system prompts.

In [ ]:
# Blueprint: Technical Screener Agent
technical_screener_blueprint = {
    "system_prompt": (
        "You are a rigorous technical interviewer specializing in distributed systems. "
        "Evaluate the candidate's technical skills based on their resume. "
        "Focus on: system design, algorithms, data structures, and cloud architecture. "
        "Score each dimension from 0 to 1 and return a JSON object with "
        "keys: technical_score, strengths, gaps, recommendation."
    ),
    "model": "gpt-4o",
    "temperature": 0.3,   # low temperature for consistent evaluation
    "max_tokens": 600,
}

# Blueprint: Cultural Fit Assessor Agent
culture_fit_blueprint = {
    "system_prompt": (
        "You are an organizational psychologist assessing cultural fit. "
        "Based on the candidate resume, infer their likely work style, "
        "collaboration preferences, and alignment with a fast-paced startup culture. "
        "Return JSON with keys: culture_score, collaboration_style, growth_mindset, risk."
    ),
    "model": "gpt-4o",
    "temperature": 0.5,
    "max_tokens": 400,
}

print("Blueprint agents defined:")
print(f"  technical_screener : {len(technical_screener_blueprint['system_prompt'])} char prompt")
print(f"  culture_fit        : {len(culture_fit_blueprint['system_prompt'])} char prompt")

## 2. Build a Graph with Blueprint Nodes

Instead of `.agent("AgentName")`, we call `.blueprint({...})` on the node builder.

Note: a node must have **either** `.agent()` **or** `.blueprint()`, not both.

In [ ]:
graph_def = (
    GraphBuilder()
    # Standard registered agent — parses raw resume text
    .node("parse_resume")
        .agent("ResumeParserAgent")
        .params(output_format="json")
        .timeout(30)
        .done()
    # Blueprint node — dynamic technical screener (no pre-registered agent required)
    .node("technical_screen")
        .blueprint(technical_screener_blueprint)   # <-- blueprint-based agent
        .params(evaluation_framework="STAR")
        .timeout(45)
        .done()
    # Blueprint node — dynamic culture fit assessor
    .node("culture_fit")
        .blueprint(culture_fit_blueprint)          # <-- blueprint-based agent
        .params(company_stage="Series B")
        .timeout(30)
        .done()
    # Standard registered agent — aggregates scores
    .node("final_score")
        .agent("ScoreAggregatorAgent")
        .params(weights={"technical": 0.6, "culture": 0.4})
        .timeout(20)
        .done()
    .edge("parse_resume",   "technical_screen")
    .edge("parse_resume",   "culture_fit")        # parallel branch
    .edge("technical_screen", "final_score")
    .edge("culture_fit",      "final_score")
    .entry("parse_resume")
    .max_cycles(10)
    .build()
)

print("Graph nodes:")
for n in graph_def["nodes"]:
    node_type = "blueprint" if "blueprint" in n else "registered agent"
    name = n.get("agent", f"<blueprint: {str(n.get('blueprint', {}).get('model', '?'))}>")
    print(f"  {n['id']:20s} [{node_type:16s}] → {name}")

## 3. The `create_agent_activity` Flow

When the orchestrator encounters `technical_screen`, internally it:

```
WorkflowTask: execute_node("technical_screen")
        │
        ▼
  node.blueprint is not None?
        │ YES
        ▼
  create_agent_activity(blueprint)
        │ → validates schema
        │ → constructs ephemeral LLM agent with system_prompt
        │ → returns ephemeral_agent_id
        ▼
  execute_agent_activity(ephemeral_agent_id, context, params)
        │ → LLM call with system_prompt + resume context
        │ → returns structured output
        ▼
  store_node_output("technical_screen", output)
```

## 4. Run the Blueprint Workflow

In [ ]:
payload = {
    "resume_text": (
        "Emma Rodriguez, 5 years full-stack, React, Node.js, PostgreSQL, AWS. "
        "Built distributed payment system at FinTech startup, team lead of 6 engineers."
    ),
    "candidate_id": "candidate-006",
}

response = client.run_graph(graph_def=graph_def, payload=payload)
instance_id = response.instance_id
print(f"Launched — instance_id: {instance_id}")
print("Note: Blueprint nodes invoke the LLM — may take 10-30 seconds.")

## 5. Compare Blueprint Output vs Standard Agent Output

In [ ]:
def wait_for_node(client, instance_id, node_id, max_wait=60):
    deadline = time.time() + max_wait
    while time.time() < deadline:
        try:
            ns = client.get_node_state(instance_id, node_id)
            if ns.output:
                return ns
        except Exception:
            pass
        time.sleep(2)
    return None

all_nodes = ["parse_resume", "technical_screen", "culture_fit", "final_score"]
results = {}

for node_id in all_nodes:
    print(f"Waiting for '{node_id}'...", end=" ")
    ns = wait_for_node(client, instance_id, node_id)
    if ns:
        results[node_id] = ns.output
        print("done")
    else:
        print("TIMEOUT")

print("\n" + "=" * 60)
print("STANDARD AGENT OUTPUT: ResumeParserAgent (parse_resume)")
print("=" * 60)
pprint.pprint(results.get("parse_resume", {}), indent=2)

print("\n" + "=" * 60)
print("BLUEPRINT AGENT OUTPUT: technical_screen")
print("=" * 60)
pprint.pprint(results.get("technical_screen", {}), indent=2)

print("\n" + "=" * 60)
print("BLUEPRINT AGENT OUTPUT: culture_fit")
print("=" * 60)
pprint.pprint(results.get("culture_fit", {}), indent=2)

## 6. Blueprint vs Static Agent — Key Differences

| Aspect | Static Agent | Blueprint Agent |
|--------|-------------|----------------|
| Registration | Must be deployed to worker | Defined inline in node spec |
| Behaviour | Code-defined | Prompt-defined (LLM) |
| Performance | Fast (no LLM overhead) | Slower (LLM call required) |
| Flexibility | Fixed at deploy time | Changes with blueprint |
| Debugging | Code-traceable | Prompt/LLM output inspection |
| Cost | No LLM tokens | Consumes LLM tokens |

**When to use blueprints:**
- Rapid iteration on agent behaviour
- Workflows that generate their own sub-agents (meta-cognitive patterns)
- Multi-tenant customization where each tenant has unique instructions

In [ ]:
metrics = client.get_metrics(instance_id)
print(f"Final metrics:")
print(f"  nodes_executed : {metrics.nodes_executed}")
print(f"  error_count    : {metrics.error_count}")

client.close()
print("\nClient closed.")

---

## Summary

```python
# Blueprint node — no pre-registered agent required
.node("my_dynamic_node")
    .blueprint({
        "system_prompt": "You are a specialized...",
        "model": "gpt-4o",
        "temperature": 0.3,
    })
    .params(...)  # extra params passed to the agent as context
    .done()
```

**Next**: See `08_observability.ipynb` for full observability — health polling, metrics, state snapshots, and charts.